# 🛰️ GEOTRAZA — Entrenamiento en Google Colab
**Taller 3 · Cartografía Automatizada** — Electiva IV Deep Computer Vision, USTA Bucaramanga

Este notebook entrena **Mask2Former** y **EfficientPS** para segmentación panóptica de imágenes satelitales (Área Caminable, Área Construida, Zona Verde, Carretera), los evalúa con IoU, mAP, F1, precisión y PQ, y genera todo el material del informe.

**Antes de empezar**
1. `Entorno de ejecución → Cambiar tipo de entorno → GPU` (T4 sirve; A100/L4 entrena ~3× más rápido).
2. Sube `cartografia-automatizada.zip` a la carpeta **DEEP COMPUTER** de tu Google Drive (*Mi unidad › DEEP COMPUTER*).
3. Ten a mano tu `kaggle.json` (Kaggle → Settings → API → Create New Token).

Los checkpoints se guardan en `DEEP COMPUTER/carto_ckpt` cada época: si Colab se desconecta, vuelve a ejecutar las celdas y el entrenamiento **continúa donde iba** (`--resume`).

Al final descargas `models/weights/*.pt` + `outputs/` y los copias al proyecto en tu PC para correr la app web.

## 1 · Entorno

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
CARPETA = f'{DRIVE}/DEEP COMPUTER'                 # carpeta de Drive donde está el zip y donde se guarda todo
ZIP = f'{CARPETA}/cartografia-automatizada.zip'
CKPT = f'{CARPETA}/carto_ckpt'                     # checkpoints persistentes
PROJ = '/content/cartografia-automatizada'

import os, glob
if not os.path.exists(ZIP):                        # si no está ahí, lo busca en todo Drive
    found = glob.glob(f'{DRIVE}/**/cartografia-automatizada*.zip', recursive=True)
    assert found, 'No encontré cartografia-automatizada.zip en tu Drive'
    ZIP = found[0]; CARPETA = os.path.dirname(ZIP); CKPT = f'{CARPETA}/carto_ckpt'
print('Zip:', ZIP)
os.makedirs(CKPT, exist_ok=True)
!rm -rf {PROJ} && unzip -q -o "{ZIP}" -d /content/
%cd {PROJ}
!ls

In [ ]:
!pip install -q -r requirements-colab.txt

## 2 · Descarga del dataset (Kaggle)

In [ ]:
# Opción A: sube tu kaggle.json con el selector de archivos
import json, os
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    up = files.upload()                      # elige kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(up['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
# Opción B (como en el notebook del profe): escribe el token a mano
# json.dump({"username": "TU_USUARIO", "key": "TU_KEY"}, open('/root/.kaggle/kaggle.json', 'w'))

In [ ]:
!kaggle datasets download -d cetaamb/reto-1-cartografa-automatizada -p data/raw -q
!cd data/raw && unzip -q -o reto-1-cartografa-automatizada.zip && rm reto-1-cartografa-automatizada.zip
!ls data/raw; ls "data/raw/Dataset segmentacion" | head

## 3 · Preprocesamiento y exploración de datos
`data/prepare_data.py` crea la división **train / val / test (70 / 15 / 15, semilla 42)**, calcula estadísticas y guarda figuras.

Decisiones de preprocesamiento (ver `data/dataset.py`):
- El GT trae 4 máscaras binarias que **se solapan** (p. ej. copas de árboles sobre andenes). Se resuelve con prioridad *Zona Verde > Carretera > Caminable > Construida* → una etiqueta por píxel; los píxeles sin etiqueta son **Fondo**.
- **Panóptico**: *things* = Área Construida y Zona Verde (1 instancia por región conexa); *stuff* = Caminable, Carretera, Fondo.
- **Aumento de datos**: escala aleatoria 0.75–1.5, recorte 512×512, 8 simetrías D4 (vista cenital sin orientación), jitter de color y desenfoque leve.

In [ ]:
!python data/prepare_data.py --figures 3

In [ ]:
import json, glob
from IPython.display import Image, display
print(json.dumps(json.load(open('outputs/dataset_stats.json')), indent=2, ensure_ascii=False))
for f in sorted(glob.glob('outputs/ejemplo_gt_*.png')):
    display(Image(f))

In [ ]:
# Muestra del aumento de datos (lo que realmente ve la red en entrenamiento)
import numpy as np, matplotlib.pyplot as plt, config as C
from data.dataset import CartografiaDataset, load_splits
from utils.visualization import colorize_semantic
ds = CartografiaDataset(load_splits()['train'][:4], train=True, crop_size=512, crops_per_image=2)
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for k in range(4):
    x, y, sid = ds[k]
    img = (x.numpy().transpose(1,2,0) * np.array(C.IMAGENET_STD) + np.array(C.IMAGENET_MEAN)).clip(0,1)
    ax[0,k].imshow(img); ax[0,k].set_title(f'recorte_{sid} aumentado'); ax[0,k].axis('off')
    ax[1,k].imshow(colorize_semantic(np.where(y.numpy()==255, 0, y.numpy()))); ax[1,k].axis('off')
plt.tight_layout(); plt.savefig('outputs/aumento_datos.png', dpi=100); plt.show()

### 3.1 · Recuperar lo que ya está en Drive (si es una sesión nueva)
Si ya entrenaste algún modelo en otra sesión, esta celda trae de `DEEP COMPUTER/carto_ckpt` los mejores pesos y el historial de pérdidas. Si es la primera vez, no hace nada.

In [ ]:
import os, shutil, json, torch
os.makedirs('models/weights', exist_ok=True); os.makedirs('outputs', exist_ok=True)
for m in ['mask2former', 'efficientps']:
    if os.path.exists(f'{CKPT}/{m}_best.pt') and not os.path.exists(f'models/weights/{m}_best.pt'):
        shutil.copy(f'{CKPT}/{m}_best.pt', f'models/weights/{m}_best.pt'); print('pesos recuperados:', m)
    h = f'outputs/history_{m}.json'
    if not os.path.exists(h):
        if os.path.exists(f'{CKPT}/history_{m}.json'):
            shutil.copy(f'{CKPT}/history_{m}.json', h)
        elif os.path.exists(f'{CKPT}/{m}_last.pt'):
            ck = torch.load(f'{CKPT}/{m}_last.pt', map_location='cpu', weights_only=False)
            json.dump(ck['history'], open(h, 'w')); print(f'historial recuperado: {m} (época {ck["epoch"] + 1})')
            del ck
    for f in os.listdir(CKPT):
        if f.startswith(f'metrics_{m}') and not os.path.exists(f'outputs/{f}'):
            shutil.copy(f'{CKPT}/{f}', f'outputs/{f}')
!ls -lh models/weights outputs

## 4 · Entrenamiento de Mask2Former
Backbone **Swin-B preentrenado en COCO-panoptic** (transfer learning), AdamW, LR 1e-4 (backbone ×0.1), warmup + decaimiento polinomial, AMP fp16. Se evalúa en validación cada 5 épocas y se guarda el mejor modelo según (mIoU + mF1 + mAP)/3.

- T4 (16 GB): `--batch-size 4` debería caber. Si sale *CUDA out of memory* usa `--batch-size 2 --accum 2`.
- Si ves muchas pérdidas no finitas, agrega `--no-amp`.
- Si Colab se desconecta: vuelve a ejecutar las celdas 1–2 y esta misma celda (tiene `--resume`).

**Si Mask2Former ya terminó en otra sesión, sáltate la celda siguiente.**

In [ ]:
!python train.py --model mask2former --epochs 80 --batch-size 4 --ckpt-dir "{CKPT}" --resume

## 5 · Entrenamiento de EfficientPS
EfficientNet-B5 (Noisy Student) + **2-way FPN** + cabeza semántica (LSFE, DPC, MC) + cabeza de instancias Mask R-CNN + **fusión panóptica**. Pérdida semántica con *bootstrapping* (25 % de píxeles más difíciles) + Dice, con **peso ×2 para Área Caminable** (la clase más difícil: delgada y poco frecuente).

Si falta memoria: `--backbone tf_efficientnet_b4.ns_jft_in1k` o `--batch-size 2 --accum 2`.

In [ ]:
!python train.py --model efficientps --epochs 100 --batch-size 4 --class-weights 1,2,1,1,1 --ckpt-dir "{CKPT}" --resume

## 6 · Evaluación en test y comparación
`train.py` ya evaluó cada modelo en test en dos modos:
- **nativo**: cabeza panóptica propia del modelo (consultas en Mask2Former, fusión en EfficientPS).
- **TTA ×8**: promedio de probabilidades sobre 8 rotaciones/espejos; instancias por regiones conexas.

Para reproducir las métricas en cualquier momento: `python evaluate.py --weights models/weights/<modelo>_best.pt --mode tta`

In [ ]:
# Si el entrenamiento se interrumpió justo al final, recupera los mejores pesos desde Drive y re-evalúa
import os, shutil
for m in ['mask2former', 'efficientps']:
    dst = f'models/weights/{m}_best.pt'
    if not os.path.exists(dst) and os.path.exists(f'{CKPT}/{m}_best.pt'):
        shutil.copy(f'{CKPT}/{m}_best.pt', dst)
    if os.path.exists(dst) and not os.path.exists(f'outputs/metrics_{m}_test_tta.json'):
        for mode in ['native', 'tta']:
            !python evaluate.py --weights {dst} --mode {mode}

### 6.1 · Ajuste del post-procesamiento en validación (sube IoU y mAP sin reentrenar)
Busca en el split de **validación** (nunca en test):
- **Semántica (TTA):** una o dos escalas (1.0 y 1.5), suavizado de probabilidades y un factor para Área Caminable.
- **Instancias:** erosión para separar manzanas unidas, área mínima de región y puntaje de ranking.

Luego re-evalúa test con esos parámetros. Tarda unos 20–40 min.

In [ ]:
!python tune_postprocess.py
for m in ['mask2former', 'efficientps']:
    if os.path.exists(f'models/weights/{m}_best.pt'):
        for mode in ['native', 'tta']:
            !python evaluate.py --weights models/weights/{m}_best.pt --mode {mode}

In [ ]:
!python report_figures.py --n 5

In [ ]:
import pandas as pd
display(pd.read_csv('outputs/tabla_comparativa.csv'))
display(pd.read_csv('outputs/tabla_por_clase.csv'))
for f in ['curvas_mask2former.png', 'curvas_efficientps.png', 'curvas_comparadas.png', 'comparacion_metricas.png']:
    if os.path.exists(f'outputs/{f}'): display(Image(f'outputs/{f}'))
for f in sorted(glob.glob('outputs/confusion_*.png')) + sorted(glob.glob('outputs/cualitativo_*.png')):
    display(Image(f))

## 7 · Exportar al PC
Descarga el zip (o cópialo desde Drive) y descomprímelo **dentro** de la carpeta del proyecto en tu PC: deja `models/weights/*.pt`, `outputs/` y `data/splits.json` en su sitio. Luego en el PC: `python main.py`.

In [ ]:
!zip -q -r carto1_resultados.zip models/weights/mask2former_best.pt models/weights/efficientps_best.pt outputs data/splits.json
!cp carto1_resultados.zip "{CARPETA}/"
!ls -lh carto1_resultados.zip
from google.colab import files
files.download('carto1_resultados.zip')

## 8 · (Opcional) Probar la app web aquí mismo en Colab
Útil si quieres verla antes de pasarla al PC.

In [ ]:
import subprocess, time
proc = subprocess.Popen(['python', 'main.py', '--port', '8000'])
time.sleep(25)
from google.colab.output import eval_js
print('Abre:', eval_js("google.colab.kernel.proxyPort(8000)"))